In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import json
import os

# Charger les données propres
df = pd.read_csv('../processed/dvf_clean.csv')
print(f"✅ Données chargées : {len(df):,} lignes")

# Colonnes utilisées pour prédire
FEATURES = [
    'surface_reelle_bati',
    'nombre_pieces_principales',
    'surface_terrain',
    'dept_num',
    'longitude',
    'latitude',
    'annee',
    'trimestre',
    'prix_m2_ref_zone'
]

TARGET = 'valeur_fonciere'

# Créer le dossier models
os.makedirs('../models', exist_ok=True)

# ============================================
# FONCTION D'ENTRAÎNEMENT
# ============================================
def entrainer_modele(df, type_bien):
    print(f"\n{'='*50}")
    print(f"ENTRAÎNEMENT — {type_bien}")
    print(f"{'='*50}")
    
    # Filtrer par type
    subset = df[df['type_local'] == type_bien].copy()
    subset = subset.sort_values('annee')  # tri chronologique
    print(f"Nombre de biens : {len(subset):,}")
    
    # Séparer X et y
    X = subset[FEATURES]
    y = subset[TARGET]
    
    # Split 80% train / 20% test (chronologique)
    split = int(len(subset) * 0.8)
    X_train, X_test = X.iloc[:split], X.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]
    
    print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")
    
    # ---- Les 3 modèles ----
    modeles = {
        'Linear (baseline)': LinearRegression(),
        'Random Forest'    : RandomForestRegressor(
                                n_estimators=100, 
                                random_state=42, 
                                n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(
                                n_estimators=100, 
                                random_state=42)
    }
    
    resultats = {}
    meilleur_modele = None
    meilleur_r2     = -np.inf
    meilleur_nom    = ""
    
    for nom, modele in modeles.items():
        print(f"\n  Entraînement {nom}...")
        modele.fit(X_train, y_train)
        y_pred = modele.predict(X_test)
        
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)
        
        resultats[nom] = {
            'RMSE': round(rmse, 0),
            'MAE' : round(mae, 0),
            'R2'  : round(r2, 4)
        }
        
        print(f"    RMSE : {rmse:,.0f} €")
        print(f"    MAE  : {mae:,.0f} €")
        print(f"    R²   : {r2:.4f}")
        
        if r2 > meilleur_r2:
            meilleur_r2     = r2
            meilleur_modele = modele
            meilleur_nom    = nom
    
    print(f"\n🏆 Meilleur modèle : {meilleur_nom} (R²={meilleur_r2:.4f})")
    
    # Sauvegarder le meilleur modèle
    type_slug = type_bien.lower().replace(' ', '_')
    chemin_modele  = f'../models/model_{type_slug}_v1.joblib'
    chemin_metrics = f'../models/metrics_{type_slug}_v1.json'
    
    joblib.dump(meilleur_modele, chemin_modele)
    
    with open(chemin_metrics, 'w') as f:
        json.dump(resultats, f, indent=2)
    
    print(f"✅ Modèle sauvegardé : {chemin_modele}")
    print(f"✅ Métriques sauvegardées : {chemin_metrics}")
    
    return meilleur_modele, resultats

# ============================================
# ENTRAÎNER LES 2 MODÈLES
# ============================================
modele_appart, metrics_appart = entrainer_modele(df, 'Appartement')
modele_maison, metrics_maison = entrainer_modele(df, 'Maison')

print("\n" + "="*50)
print("✅ ENTRAÎNEMENT TERMINÉ")
print("="*50)

✅ Données chargées : 56,719 lignes

ENTRAÎNEMENT — Appartement
Nombre de biens : 51,628
Train : 41,302 | Test : 10,326

  Entraînement Linear (baseline)...
    RMSE : 323,016 €
    MAE  : 168,607 €
    R²   : 0.6915

  Entraînement Random Forest...
    RMSE : 273,235 €
    MAE  : 141,932 €
    R²   : 0.7793

  Entraînement Gradient Boosting...
    RMSE : 281,508 €
    MAE  : 144,912 €
    R²   : 0.7657

🏆 Meilleur modèle : Random Forest (R²=0.7793)
✅ Modèle sauvegardé : ../models/model_appartement_v1.joblib
✅ Métriques sauvegardées : ../models/metrics_appartement_v1.json

ENTRAÎNEMENT — Maison
Nombre de biens : 5,091
Train : 4,072 | Test : 1,019

  Entraînement Linear (baseline)...
    RMSE : 1,068,109 €
    MAE  : 314,783 €
    R²   : -2.1632

  Entraînement Random Forest...
    RMSE : 382,193 €
    MAE  : 219,123 €
    R²   : 0.5950

  Entraînement Gradient Boosting...
    RMSE : 400,050 €
    MAE  : 228,842 €
    R²   : 0.5563

🏆 Meilleur modèle : Random Forest (R²=0.5950)
✅ Modèle 

In [2]:
# Vérification avant de créer l'app
import joblib
import os

# Vérifier que les modèles existent
modeles = [
    '../models/model_appartement_v1.joblib',
    '../models/model_maison_v1.joblib'
]

for m in modeles:
    if os.path.exists(m):
        taille = os.path.getsize(m) / 1024 / 1024
        print(f"✅ {m} ({taille:.1f} Mo)")
    else:
        print(f"❌ MANQUANT : {m}")

# Tester une prédiction
print("\n=== TEST DE PRÉDICTION ===\n")

modele_appart = joblib.load('../models/model_appartement_v1.joblib')
modele_maison = joblib.load('../models/model_maison_v1.joblib')

# Exemple : Appartement 65m², 3 pièces, Paris 75
exemple_appart = [[
    65,     # surface
    3,      # pièces
    0,      # terrain (0 pour appart)
    75,     # département
    2.35,   # longitude Paris
    48.85,  # latitude Paris
    2024,   # année
    2,      # trimestre
    9500    # prix_m2_ref_zone Paris
]]

prix_predit = modele_appart.predict(exemple_appart)[0]
print(f"Appartement 65m², 3 pièces, Paris 75 :")
print(f"  → Prix estimé : {prix_predit:,.0f} €")
print(f"  → Prix au m²  : {prix_predit/65:,.0f} €/m²")

# Exemple : Maison 120m², 5 pièces, Val-de-Marne 94
exemple_maison = [[
    120,    # surface
    5,      # pièces
    300,    # terrain
    94,     # département
    2.45,   # longitude
    48.78,  # latitude
    2024,   # année
    2,      # trimestre
    3800    # prix_m2_ref_zone 94
]]

prix_predit_maison = modele_maison.predict(exemple_maison)[0]
print(f"\nMaison 120m², 5 pièces, terrain 300m², Val-de-Marne 94 :")
print(f"  → Prix estimé : {prix_predit_maison:,.0f} €")
print(f"  → Prix au m²  : {prix_predit_maison/120:,.0f} €/m²")

✅ ../models/model_appartement_v1.joblib (335.9 Mo)
✅ ../models/model_maison_v1.joblib (34.3 Mo)

=== TEST DE PRÉDICTION ===



C:\Users\SuperElectro\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


Appartement 65m², 3 pièces, Paris 75 :
  → Prix estimé : 865,069 €
  → Prix au m²  : 13,309 €/m²

Maison 120m², 5 pièces, terrain 300m², Val-de-Marne 94 :
  → Prix estimé : 861,000 €
  → Prix au m²  : 7,175 €/m²


C:\Users\SuperElectro\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
